# nested-param-group-loop — worked example 1: manual SGD via the nested param_groups loop

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nested-param-group-loop`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

An optimizer stores its parameters as `param_groups`, a `list[dict]` where each dict has a `'params'` list plus per-group hyperparameters like `'lr'`. The canonical update is a double loop: outer over groups (read the group's hparams once), inner over that group's params (skip any whose `.grad` is `None`).

## Worked solution

We build a real `torch.optim.SGD` over two parameter groups with different learning rates, give the parameters fake gradients, then reimplement the step by hand. The outer loop walks `optimizer.param_groups` and reads `lr = group['lr']` once per group. The inner loop walks `group['params']`; we `continue` past any param with `p.grad is None` because subtracting `None` is impossible and such a param simply did not participate. For the rest we update in place with `p.data.add_(p.grad, alpha=-lr)`, which is `p -= lr * grad` without building autograd history. We seed, snapshot the original values, run the manual step, and print the change for the first parameter to confirm it moved by `-lr * grad`.

In [ ]:
import torch as t

t.manual_seed(0)

a = t.nn.Parameter(t.randn(3))
b = t.nn.Parameter(t.randn(2, 2))
opt = t.optim.SGD([
    {'params': [a], 'lr': 0.1},
    {'params': [b], 'lr': 0.5},
])
# fake gradients
a.grad = t.ones(3)
b.grad = t.full((2, 2), 2.0)

before_a = a.data.clone()

def manual_sgd_step(optimizer):
    for group in optimizer.param_groups:
        lr = group['lr']
        for p in group['params']:
            if p.grad is None:
                continue
            p.data.add_(p.grad, alpha=-lr)

manual_sgd_step(opt)
print('delta a:', (a.data - before_a))  # should be -0.1 each